# 18 — Exact & Deterministic Neural Network Verification — Lab

## Description

This lab exercises Interval Bound Propagation (IBP) and $\alpha$-CROWN certification methods using the auto_LiRPA library. You will load a pre-trained 3-layer neural network trained on MNIST, compute certified robustness bounds under $L_\infty$ perturbations, and compare the tightness of IBP versus $\alpha$-CROWN on 100 test examples. This hands-on experience demonstrates how deterministic verification methods work in practice and when to prefer speed (IBP) over tightness ($\alpha$-CROWN).

## Objectives

- Student can trace Interval Bound Propagation through a two-layer ReLU network by hand, propagating input interval to output interval.
- Student can use auto_LiRPA to obtain IBP and $\alpha$-CROWN certified bounds for a small neural network under an $L_\infty$ perturbation.
- Student can state the completeness and soundness guarantees of SMT-based verification (Reluplex) versus relaxation-based methods.
- Student can explain why IBP is sound but incomplete and why this matters for certification claims.
- Student can describe the scalability limits of complete verification and when incomplete methods are preferred.

## Prerequisites

Before starting this lab, you should be comfortable with: neural network forward passes and ReLU activations (from Lesson 03 and Lesson 04), the formal verification problem statement and certified intervals (from this lesson's lecture), basic PyTorch tensor operations and model evaluation, and working with the MNIST dataset.

## Required Software and Packages

- PyTorch (version 1.9 or later)
- auto_LiRPA (install via pip install auto_LiRPA)
- torchvision (for MNIST dataset)
- numpy
- pandas (for tabular output)

## Environment and Setup Instructions

Run the cell below to import required packages and verify that auto_LiRPA is installed. 

**Cell 1 — Dependency check.** This cell makes sure the `auto_LiRPA` library is available before the lab begins. It first tries to import the package; if the import succeeds, nothing is installed. If it fails, the cell installs the library from PyPI with `%pip install auto_LiRPA`. A commented-out alternative installs the latest release directly from GitHub, which is useful because the PyPI release pins an older PyTorch version (`torch<1.13`) and may conflict with newer environments.

In [1]:
# Install auto_LiRPA if it is not already available in this environment
try:
    import auto_LiRPA
    print('auto_LiRPA is already installed.')
except ImportError:
    %pip install --quiet auto_LiRPA
    # Note: the PyPI release pins torch<1.13. If the install above conflicts
    # with your PyTorch version, install the latest release from GitHub instead:
    # %pip install --quiet "git+https://github.com/Verified-Intelligence/auto_LiRPA"
    import auto_LiRPA
    print('auto_LiRPA installed.')

/Users/mohabuhamad/Downloads/comp378_f26/.venv/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


auto_LiRPA is already installed.


**Cell 2 — Environment setup, model definition, and training.** This cell imports PyTorch, torchvision, numpy, pandas, and the three key auto_LiRPA classes (`BoundedModule`, `BoundedTensor`, `PerturbationLpNorm`). It then defines a 3-layer MLP ($784 \rightarrow 512 \rightarrow 256 \rightarrow 10$) with ReLU activations for MNIST classification. The MNIST test set is loaded with standard normalization (mean 0.1307, std 0.3081) and the first 100 test examples are collected for certification. If a saved checkpoint (`mnist_mlp.pth`) exists it is loaded; otherwise the model is quickly trained for 2 epochs on a 12,800-image subset and the weights are saved for future runs. Finally the cell switches the model to evaluation mode and reports its accuracy on the 100 test examples, so later falsified results reflect genuine misclassifications rather than an untrained network.

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import pandas as pd
import os
from auto_LiRPA import PerturbationLpNorm, BoundedModule, BoundedTensor
import warnings
warnings.filterwarnings('ignore')

# Define the neural network model
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(784, 512)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(512, 256)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(256, 10)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.fc3(x)
        return x

model = MLP()

# Load MNIST test set
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# Extract first 100 examples
images = []
labels = []
for i, (img, label) in enumerate(test_loader):
    if i >= 100:
        break
    images.append(img)
    labels.append(label)

images = torch.cat(images, dim=0)
labels = torch.cat(labels, dim=0)

# Train the model on MNIST (or load the saved checkpoint if it exists)
ckpt_path = 'mnist_mlp.pth'
if os.path.exists(ckpt_path):
    model.load_state_dict(torch.load(ckpt_path))
    print(f'Loaded pre-trained weights from {ckpt_path}')
else:
    train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    model.train()
    epochs = 2
    for epoch in range(epochs):
        for i, (img, label) in enumerate(train_loader):
            if i >= 200:  # train on a 12,800-image subset to keep this fast on CPU
                break
            optimizer.zero_grad()
            loss = criterion(model(img), label)
            loss.backward()
            optimizer.step()
        print(f'Epoch {epoch + 1}/{epochs} - last batch loss: {loss.item():.4f}')
    torch.save(model.state_dict(), ckpt_path)
    print(f'Saved trained weights to {ckpt_path}')

model.eval()

with torch.no_grad():
    preds = model(images).argmax(dim=1)
print(f'Accuracy on the 100 test examples: {(preds == labels).float().mean().item() * 100:.1f}%')

print('All packages imported successfully.')
print(f'Model loaded with {sum(p.numel() for p in model.parameters())} parameters.')
print(f'Loaded {images.shape[0]} MNIST test examples.')


  0%|          | 0.00/9.91M [00:00<?, ?B/s]


  4%|▍         | 426k/9.91M [00:00<00:02, 3.92MB/s]


 19%|█▉        | 1.90M/9.91M [00:00<00:00, 10.0MB/s]


 30%|██▉       | 2.95M/9.91M [00:00<00:00, 8.86MB/s]


 51%|█████     | 5.01M/9.91M [00:00<00:00, 11.0MB/s]

...


100%|██████████| 1.65M/1.65M [00:00<00:00, 6.88MB/s]


  0%|          | 0.00/4.54k [00:00<?, ?B/s]


100%|██████████| 4.54k/4.54k [00:00<00:00, 12.9MB/s]

Epoch 1/2 - last batch loss: 0.1738


Epoch 2/2 - last batch loss: 0.1400
Saved trained weights to mnist_mlp.pth
Accuracy on the 100 test examples: 97.0%
All packages imported successfully.
Model loaded with 535818 parameters.
Loaded 100 MNIST test examples.


## Background and Context

In the lecture, you learned three verification methods: Interval Bound Propagation (IBP), linear relaxation ($\alpha$-CROWN), and SMT-based solvers (Reluplex). This lab focuses on the first two using auto_LiRPA. The library wraps a PyTorch network and overloads its forward pass to compute certified bounds. You specify a perturbation region ($L_\infty$ ball with radius $\varepsilon = 0.1$), and auto_LiRPA returns (lower_bound, upper_bound) for each output logit. If the lower bound for the true class exceeds all upper bounds for other classes, the network is certified robust.

## Exercise Instructions

1. Define a helper function to classify each example's status and run IBP certification on 100 MNIST examples under $L_\infty$ perturbation $\varepsilon = 0.1$. Record tallies for certified, uncertified-but-correct, and falsified examples. 
2. Run $\alpha$-CROWN certification on the same 100 examples using a separate BoundedModule with the alpha-CROWN method. Record tallies for all three outcome categories. 
3. Build a comparison table showing certified accuracy, uncertified-but-correct rate, and falsified rate for IBP and $\alpha$-CROWN. 
4. Answer the two comprehension questions. 

**Cell 3 — IBP certification.** The helper `classify_result` labels an example as `falsified` when the clean prediction is already wrong, `certified` when the lower bound of the true-class logit exceeds every other class's upper bound (a proof of robustness), and `uncertified-but-correct` otherwise. The loop wraps the model in an auto_LiRPA `BoundedModule`, then for each of the 100 test images: (1) makes the clean prediction, (2) defines an $L_\infty$ perturbation ball of radius $\varepsilon = 0.1$ with `PerturbationLpNorm`, (3) wraps the image as a `BoundedTensor`, and (4) calls `compute_bounds(method='IBP')`, which propagates the input interval forward through the network layer by layer to get certified lower/upper bounds on every logit. Each example is classified and the three tallies are printed.

In [3]:
# Helper function to classify prediction status
def classify_result(model_pred, true_label, lower_bounds, upper_bounds):
    """Classify example as certified, uncertified-but-correct, or falsified."""
    if model_pred != true_label:
        return 'falsified'
    true_lower = lower_bounds[true_label]
    max_other_upper = np.max(np.concatenate([upper_bounds[:true_label], upper_bounds[true_label+1:]]))
    if true_lower > max_other_upper:
        return 'certified'
    else:
        return 'uncertified-but-correct'

# IBP certification loop
eps = 0.1
ibp_results = []

# Create a BoundedModule that wraps the model for bound computation
model_ibp = BoundedModule(model, torch.zeros(1, 784))

for i in range(len(images)):
    # 1. Extract image i and its true label
    image = images[i:i + 1].reshape(1, -1)
    true_label = int(labels[i])

    # 2. Model prediction on the clean image
    model_pred = int(model(image).argmax(dim=1).item())

    # 3-4. Define the L-inf perturbation region and wrap the image as a BoundedTensor
    ptb = PerturbationLpNorm(norm=np.inf, eps=eps)
    bounded_image = BoundedTensor(image, ptb)

    # 5-6. Compute certified lower and upper bounds on all output logits
    lower, upper = model_ibp.compute_bounds(x=(bounded_image,), method='IBP')
    lower = lower.detach().cpu().numpy().flatten()
    upper = upper.detach().cpu().numpy().flatten()

    # 7. Classify the result for this example
    ibp_results.append(classify_result(model_pred, true_label, lower, upper))

# Compute tallies
ibp_certified = ibp_results.count('certified')
ibp_uncertified = ibp_results.count('uncertified-but-correct')
ibp_falsified = ibp_results.count('falsified')

print('IBP Results (eps = 0.1, 100 examples):')
print(f'  Certified:               {ibp_certified}')
print(f'  Uncertified-but-correct: {ibp_uncertified}')
print(f'  Falsified:               {ibp_falsified}')

IBP Results (eps = 0.1, 100 examples):
  Certified:               0
  Uncertified-but-correct: 97
  Falsified:               3


**Cell 4 — $\alpha$-CROWN certification.** This cell repeats the same 100-example loop using `method='alpha-CROWN'` on a separate `BoundedModule`. Instead of propagating fixed intervals forward, $\alpha$-CROWN propagates linear relaxations backward through the network and optimizes a learnable slope $\alpha$ per unstable ReLU neuron via gradient descent, producing tighter bounds at the cost of more computation (you will see occasional 'Early stop' messages from this optimization). The same `classify_result` helper and the same $\varepsilon = 0.1$ perturbation are used so the tallies are directly comparable to IBP.

In [4]:
# α-CROWN certification loop
# Create a separate BoundedModule that uses the alpha-CROWN method
model_acrown = BoundedModule(model, torch.zeros(1, 784))

acrown_results = []

for i in range(len(images)):
    # 1. Extract image i and its true label
    image = images[i:i + 1].reshape(1, -1)
    true_label = int(labels[i])

    # 2. Model prediction on the clean image
    model_pred = int(model(image).argmax(dim=1).item())

    # 3-4. Define the L-inf perturbation region and wrap the image as a BoundedTensor
    ptb = PerturbationLpNorm(norm=np.inf, eps=eps)
    bounded_image = BoundedTensor(image, ptb)

    # 5-6. Compute certified bounds with alpha-CROWN (optimizes per-neuron slopes)
    lower, upper = model_acrown.compute_bounds(x=(bounded_image,), method='alpha-CROWN')
    lower = lower.detach().cpu().numpy().flatten()
    upper = upper.detach().cpu().numpy().flatten()

    # 7. Classify the result for this example
    acrown_results.append(classify_result(model_pred, true_label, lower, upper))

# Compute tallies
acrown_certified = acrown_results.count('certified')
acrown_uncertified = acrown_results.count('uncertified-but-correct')
acrown_falsified = acrown_results.count('falsified')

print('alpha-CROWN Results (eps = 0.1, 100 examples):')
print(f'  Certified:               {acrown_certified}')
print(f'  Uncertified-but-correct: {acrown_uncertified}')
print(f'  Falsified:               {acrown_falsified}')

Early stop at 10th iter due to 10 iterations no improvement!
Early stop at 10th iter due to 10 iterations no improvement!


Early stop at 12th iter due to 10 iterations no improvement!
Early stop at 10th iter due to 10 iterations no improvement!


Early stop at 10th iter due to 10 iterations no improvement!
Early stop at 10th iter due to 10 iterations no improvement!


Early stop at 10th iter due to 10 iterations no improvement!
Early stop at 10th iter due to 10 iterations no improvement!


Early stop at 10th iter due to 10 iterations no improvement!
Early stop at 10th iter due to 10 iterations no improvement!


...

Early stop at 12th iter due to 10 iterations no improvement!
Early stop at 10th iter due to 10 iterations no improvement!


Early stop at 10th iter due to 10 iterations no improvement!
Early stop at 10th iter due to 10 iterations no improvement!


Early stop at 10th iter due to 10 iterations no improvement!
Early stop at 10th iter due to 10 iterations no improvement!


Early stop at 10th iter due to 10 iterations no improvement!
Early stop at 10th iter due to 10 iterations no improvement!


Early stop at 10th iter due to 10 iterations no improvement!
Early stop at 10th iter due to 10 iterations no improvement!
alpha-CROWN Results (eps = 0.1, 100 examples):
  Certified:               3
  Uncertified-but-correct: 94
  Falsified:               3


**Cell 5 — Comparison table and verification.** This cell converts the raw tallies from both methods into percentages and displays them in a pandas DataFrame with one row per method (IBP, $\alpha$-CROWN) and one column per outcome category. It then runs sanity assertions: each method's three tallies must sum to 100 and each results list must contain exactly 100 entries. If everything is consistent it prints `PASS`, confirming the certification pipeline ran on all 100 examples for both methods.

In [5]:
# Build comparison table
total = 100

ibp_certified_pct = (ibp_certified / total) * 100
ibp_uncertified_pct = (ibp_uncertified / total) * 100
ibp_falsified_pct = (ibp_falsified / total) * 100

acrown_certified_pct = (acrown_certified / total) * 100
acrown_uncertified_pct = (acrown_uncertified / total) * 100
acrown_falsified_pct = (acrown_falsified / total) * 100

comparison_data = {
    'Method': ['IBP', 'α-CROWN'],
    'Certified (%)': [f'{ibp_certified_pct:.1f}', f'{acrown_certified_pct:.1f}'],
    'Uncertified-but-Correct (%)': [f'{ibp_uncertified_pct:.1f}', f'{acrown_uncertified_pct:.1f}'],
    'Falsified (%)': [f'{ibp_falsified_pct:.1f}', f'{acrown_falsified_pct:.1f}']
}

comparison_df = pd.DataFrame(comparison_data)
print('\nComparison Table: IBP vs α-CROWN (100 MNIST examples, ε=0.1)')
print(comparison_df.to_string(index=False))

# Verification checks
ibp_total = ibp_certified + ibp_uncertified + ibp_falsified
acrown_total = acrown_certified + acrown_uncertified + acrown_falsified

assert ibp_total == 100, f'IBP tallies do not sum to 100: {ibp_total}'
assert acrown_total == 100, f'α-CROWN tallies do not sum to 100: {acrown_total}'
assert len(ibp_results) == 100, f'IBP list has {len(ibp_results)} entries, expected 100'
assert len(acrown_results) == 100, f'α-CROWN list has {len(acrown_results)} entries, expected 100'

print('\nPASS: All verification checks passed.')
print('Both IBP and α-CROWN successfully certified bounds for 100 examples.')


Comparison Table: IBP vs α-CROWN (100 MNIST examples, ε=0.1)
 Method Certified (%) Uncertified-but-Correct (%) Falsified (%)
    IBP           0.0                        97.0           3.0
α-CROWN           3.0                        94.0           3.0

PASS: All verification checks passed.
Both IBP and α-CROWN successfully certified bounds for 100 examples.


## Expected Outputs

The IBP and $\alpha$-CROWN cells above should each print three tallies summing to 100. IBP typically shows low certified accuracy due to loose interval bounds. $\alpha$-CROWN typically shows higher certified accuracy by optimizing per-neuron slopes. Both are sound: any certified example is truly robust within the $\varepsilon$-ball.

## Comprehension Questions

1. Why does $\alpha$-CROWN typically achieve higher certified accuracy than IBP on the same examples? Explain the algorithmic difference in how bounds propagate through layers.
2. If $\alpha$-CROWN reports that an example is certified robust, can an adversarial example exist within the $\varepsilon$-ball? Justify your answer using the concept of soundness from the lecture.

## Optional Challenges

These optional extensions deepen understanding and are not required for completion.

1. Modify $\varepsilon$ to 0.05 or 0.2 and re-run IBP and $\alpha$-CROWN. How does certified accuracy change with perturbation radius? Plot the results for $\varepsilon$ values 0.05, 0.1, 0.15, 0.2.
2. For one specific example, manually compute IBP bounds after the first layer by hand using the formula

$$
\underline{z} = W_{+}\, \underline{x} + W_{-}\, \overline{x} + b
$$

Verify your computation matches the auto_LiRPA output for that example.

## Completion Criteria

You have completed this lab when all of the following are true:
1. The IBP certification cell runs without error and prints tallies for all 100 examples.
2. The $\alpha$-CROWN certification cell runs without error and prints tallies for all 100 examples.
3. The comparison table displays percentages for both methods across all three categories (Certified, Uncertified-but-Correct, Falsified), with each row summing to 100%.
4. The verification code prints PASS, confirming that all tallies sum correctly to 100 per method.
5. You have answered both comprehension questions based on your results and the lecture concepts.